In [1]:
import pandas as pd
import numpy as np
import sklearn
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from xgboost import XGBRegressor
import os
from pathlib import Path
import yaml

In [2]:
#Load data processed
project_dir = Path.cwd().parent

# Load config from folder data
with open(f'{project_dir}\\src\\data\\config.yaml', "r") as f:
    config = yaml.safe_load(f)

# Processed configuration
fileNameProc = config['fileNames']['processed']
fileTypeProc = config['fileTypes']['processed']
pathProc = project_dir.joinpath(config['paths']['processed_data'])

In [3]:
# class TranformData(BaseEstimator, TransformerMixin):
#     def __init__(self):
#         pass

#     def fit(self, X, y=None):
#         return self

#     def transform(self, X):
#         X = X.copy()

#         return X

In [4]:
# Read data
df = pd.read_parquet(f'{pathProc}\\{fileNameProc}.parquet')
df.head()

,Origen_Id,fecha_hora,demanda,lag_1h,lag_24h,lag_7d,media_24h,hora,dia_semana,mes,dia_mes,es_fin_semana
168,10,2025-01-08 00:00:00,0,0.0,0.0,0.0,1.958333,0,2,1,8,0
169,10,2025-01-08 01:00:00,0,0.0,0.0,0.0,1.958333,1,2,1,8,0
170,10,2025-01-08 02:00:00,0,0.0,0.0,0.0,1.958333,2,2,1,8,0
171,10,2025-01-08 03:00:00,0,0.0,0.0,0.0,1.958333,3,2,1,8,0
172,10,2025-01-08 04:00:00,0,0.0,0.0,0.0,1.958333,4,2,1,8,0


In [5]:
for col in df.select_dtypes(include=["int64"]):
    df[col] = pd.to_numeric(df[col], downcast="integer")

for col in df.select_dtypes(include=["float64"]):
    df[col] = pd.to_numeric(df[col], downcast="float")

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 7147968 entries, 168 to 7227263
Data columns (total 12 columns):
 #   Column         Dtype         
---  ------         -----         
 0   Origen_Id      object        
 1   fecha_hora     datetime64[ns]
 2   demanda        int16         
 3   lag_1h         float32       
 4   lag_24h        float32       
 5   lag_7d         float32       
 6   media_24h      float32       
 7   hora           int32         
 8   dia_semana     int32         
 9   mes            int32         
 10  dia_mes        int32         
 11  es_fin_semana  int8          
dtypes: datetime64[ns](1), float32(4), int16(1), int32(4), int8(1), object(1)
memory usage: 402.2+ MB


# Test Split y transformar

In [7]:
TRAIN_SPLIT = 0.80   # 80% train, 20% test

# Ordenar por fecha
demanda_modelo = df.sort_values("fecha_hora").copy()

# Fechas únicas ordenadas: así no separas una misma hora entre train y test
fechas_unicas = demanda_modelo["fecha_hora"].drop_duplicates().sort_values()

# Posición de corte
indice_corte = int(len(fechas_unicas) * TRAIN_SPLIT)

# Fecha a partir de la cual inicia test
fecha_corte = fechas_unicas.iloc[indice_corte]

# Split temporal
train = demanda_modelo[
    demanda_modelo["fecha_hora"] < fecha_corte
].copy()

test = demanda_modelo[
    demanda_modelo["fecha_hora"] >= fecha_corte
].copy()

print("Fecha de corte:", fecha_corte)
print(f"Train: {train['fecha_hora'].min()} a {train['fecha_hora'].max()}")
print(f"Test:  {test['fecha_hora'].min()} a {test['fecha_hora'].max()}")

print(f"% train: {len(train) / len(demanda_modelo):.2%}")
print(f"% test:  {len(test) / len(demanda_modelo):.2%}")

Fecha de corte: 2026-05-27 19:00:00
Train: 2025-01-08 00:00:00 a 2026-05-27 18:00:00
Test:  2026-05-27 19:00:00 a 2026-09-30 23:00:00
% train: 80.00%
% test:  20.00%


In [8]:
# Separacion de X y y
X_train = train.drop(columns=["demanda"])
y_train = train["demanda"]

X_test = test.drop(columns=["demanda"])
y_test = test["demanda"]

# Creacion de pipeline

In [9]:
categorical_features = X_train.select_dtypes('object').columns
numeric_features = X_train.select_dtypes('number').columns

preprocessor = ColumnTransformer(
    transformers=[
        ("num",StandardScaler(), numeric_features),
        ("cat",OneHotEncoder(handle_unknown="ignore", sparse_output=True), categorical_features),
    ]
)


pipeline = Pipeline([
    ('Preprocessing', preprocessor),
    ('Model',XGBRegressor())
])

In [10]:
# pipeline.fit(X_train,y_train)

In [11]:
# from sklearn.metrics import (
#     mean_absolute_error,
#     mean_squared_error,
#     r2_score,
# )
# import numpy as np

# y_pred = pipeline.predict(X_test)

# mae = mean_absolute_error(y_test, y_pred)
# rmse = np.sqrt(mean_squared_error(y_test, y_pred))
# r2 = r2_score(y_test, y_pred)

# print(f"MAE : {mae:.2f}")
# print(f"RMSE: {rmse:.2f}")
# print(f"R²  : {r2:.4f}")

In [12]:
from sklearn.model_selection import RandomizedSearchCV
from xgboost import XGBRegressor
from sklearn.model_selection import TimeSeriesSplit

tscv = TimeSeriesSplit(n_splits=5)

pipeline = Pipeline([
    ('preprocess', preprocessor),
    ('model', XGBRegressor(objective='reg:squarederror', device="cuda"))
])

param_grid = {
    'model__n_estimators': [200, 500],
    'model__max_depth': [3, 7],
    'model__learning_rate': [0.01, 0.05, 0.1],
    'model__subsample': [0.7, 0.8, 1.0],
    'model__colsample_bytree': [0.7, 0.8, 1.0],
    'model__min_child_weight': [1, 3, 5]
}

search = RandomizedSearchCV(
    pipeline,
    param_distributions=param_grid,
    n_iter=10,
    cv=tscv,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1,
    verbose=1
)

search.fit(X_train, y_train)

Fitting 5 folds for each of 10 candidates, totalling 50 fits


KeyboardInterrupt: 

In [ ]:
best_pipeline = search.best_estimator_

y_pred = best_pipeline.predict(X_test)